# E-commerce Sales & Customer Analysis

**Portfolio Project | Python • SQL • BigQuery • Tableau**

## Project overview

This project analyzes e-commerce sessions, sales, customers, traffic channels, devices, geography and product categories.

The analysis covers:
- data extraction and preparation;
- sales and customer analysis;
- sales dynamics and seasonality;
- traffic-channel performance;
- device and geographic analysis;
- statistical tests and relationships between variables;
- business insights and recommendations.

**Data period:** 1 November 2020 – 31 January 2021  
**Dataset size:** 349,545 rows and 19 columns.

> **Portfolio note:** The original project used a training BigQuery project (`data-analytics-mate`). That source is no longer accessible from the current account, so the portfolio version keeps the original SQL visible for transparency and uses a local cleaned CSV as the intended reproducible data source. The analytical conclusions below are preserved from the completed project.

## Key findings

- Americas is the leading region by sales.
- United States is the leading country by revenue.
- The `Nest` category is the largest revenue contributor.
- Desktop generates the majority of revenue.
- Direct and Organic Search are the dominant traffic channels.
- Sales show clear weekly and holiday seasonality.
- Daily sessions and sales have a strong positive correlation (`r = 0.791`, `p < 0.05`).
- Organic traffic shares in Europe and Americas were statistically similar (`p = 0.7722`).
- Device type and registration status were not statistically significantly associated (`p = 0.2318`).
- The project also includes non-parametric group comparisons and proportion tests.

## Tools

- **Python:** pandas, NumPy, SciPy, statsmodels, Matplotlib, Seaborn
- **SQL:** BigQuery
- **Visualization:** Tableau Public


In [ ]:
# Portfolio version: load the cleaned dataset locally.
# Place cleaned_project_data.csv in the same folder as this notebook.
import pandas as pd

DATA_PATH = 'cleaned_project_data.csv'
df_project = pd.read_csv(DATA_PATH)
df_project['order_date'] = pd.to_datetime(df_project['order_date'])

print(f"Дані успішно завантажені: {df_project.shape[0]:,} рядків, {df_project.shape[1]} колонок")
df_project.head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

2. SQL запит:

### Original BigQuery SQL used to build the analytical dataset

The query below documents the original data preparation step. It is kept as a portfolio reference and is **not executed** in the local reproducible version.

```sql
SELECT
    s.date AS order_date,                       -- дата сесії/замовлення з таблиці session
    sp.ga_session_id,                           -- ідентифікатор сесії
    sp.continent,                               -- континент
    sp.country,                                 -- країна
    sp.device,                                  -- девайс
    sp.browser,                                 -- браузер
    sp.mobile_model_name,                       -- назва моделі пристрою
    sp.operating_system,                        -- операційна система
    sp.language,                                -- мова браузера
    sp.medium,                                  -- тип трафіку
    sp.name AS source,                          -- джерело трафіку
    sp.channel,                                 -- канал трафіку
    acc.id AS user_id,                          -- ідентифікатор зареєстрованого користувача
    acc.is_verified AS is_email_confirmed,      -- чи підтверджено email (1/0)
    acc.is_unsubscribed,                        -- чи відписався від розсилки (1/0)
    p.category AS category_name,                -- категорія товару
    p.name AS product_name,                     -- назва товару
    p.price,                                    -- ціна товару
    p.short_description AS product_description  -- короткий опис товару
FROM `data-analytics-mate.DA.session_params` sp

-- 1. Підтягуємо дату сесії з таблиці session
LEFT JOIN `data-analytics-mate.DA.session` s
    ON sp.ga_session_id = s.ga_session_id

-- 2. Зв'язуємо сесію з акаунтом через проміжну таблицю account_session
LEFT JOIN `data-analytics-mate.DA.account_session` acts
    ON sp.ga_session_id = acts.ga_session_id

-- 3. Підтягуємо дані про користувача з таблиці account
LEFT JOIN `data-analytics-mate.DA.account` acc
    ON acts.account_id = acc.id

-- 4. Об'єднуємо з таблицею order для виявлення куплених товарів за сесію
LEFT JOIN `data-analytics-mate.DA.order` o
    ON sp.ga_session_id = o.ga_session_id

-- 5. Додаємо характеристики товару з таблиці product
LEFT JOIN `data-analytics-mate.DA.product` p
    ON o.item_id = p.item_id
```


3. Очищення даних:

In [ ]:
# Перетворюємо колонку з датою у правильний формат
df_project['order_date'] = pd.to_datetime(df_project['order_date'])

In [ ]:
#працюємо з пропусками
# 1. Для категоріальних текстових колонок (назва товару, категорія) заповнюємо як "Not Purchased"
product_cols = ['category_name', 'product_name', 'product_description']
df_project[product_cols] = df_project[product_cols].fillna('Not Purchased')

# 2. Для ціни (price) пропуски логічно замінити на 0 (бо покупки не було)
df_project['price'] = df_project['price'].fillna(0.0)

# 3. Для статусів користувачів (якщо вони незареєстровані) можна замінити на -1 або окремий маркер
user_cols = ['user_id', 'is_email_confirmed', 'is_unsubscribed']
df_project[user_cols] = df_project[user_cols].fillna(-1)

In [ ]:
#опрацьовуємо невизначені значення
# Заміняємо текстові "(not set)" на "Unknown" у всьому датасеті або в конкретних колонках
df_project = df_project.replace('(not set)', 'Unknown')
df_project = df_project.replace('None', 'Unknown') # якщо десь є текст "None"

In [ ]:
# опрацьовуємо дублікати
# Перевіряємо кількість повних дублікатів
print(f"Знайдено повних дублікатів: {df_project.duplicated().sum()}")

# Видаляємо їх, якщо вони є
df_project = df_project.drop_duplicates().reset_index(drop=True)

In [ ]:
# Перевіряємо, чи є ціни менше 0
negative_prices = df_project[df_project['price'] < 0]
print(f"Рядків з від'ємною ціною: {len(negative_prices)}")

3. Опис датасету:

In [ ]:
# Подивимось на загальну кількість рядків, колонок та типи даних
df_project.info()



- Отриманий датасет містить загалом 19 колонок (полів) та охоплює 349 545 рядків (записів).

- У датасеті присутні 5 колонок числового типу:
Цілі числа (Int64): 4 колонки. Це технічні ідентифікатори та бінарні мітки (прапорці):
ga_session_id — ідентифікатор сесії.
user_id — унікальний номер зареєстрованого користувача.
is_email_confirmed — статус підтвердження пошти (1 або 0).
is_unsubscribed — статус відписки від розсилки (1 або 0).
Дійсні числа / числа з плаваючою крапкою (float64): 1 колонка:
price — ціна товару.

- До категоріального (текстового) типу object належать 13 колонок. Вони містять демографічні дані, технічні характеристики пристроїв відвідувачів, інформацію про маркетинг, а також описи товарів:
Географія: continent, country.
Технічні параметри сесії: device, browser, mobile_model_name, operating_system, language.
Маркетингові канали: medium, source, channel.
Товарна номенклатура: category_name, product_name, product_description.

- У датасеті є 1 колонка типу дат (datetime64[ns]):
order_date — дата здійснення сесії користувача (або потенційного замовлення)

In [ ]:
print("Кількість унікальних сесій:", df_project['ga_session_id'].nunique())
print("Мінімальна дата (від):", df_project['order_date'].min())
print("Максимальна дата (до):", df_project['order_date'].max())

- Кількість унікальних сесій у датасеті становить 349 545.

- Дані охоплюють точний період у 3 місяці:
Початок періоду: 1 листопада 2020 року (2020-11-01)
Кінець періоду: 31 січня 2021 року (2021-01-31)

- Так, у датасеті є пропущені значення. Проте, завдяки перевірці через info(), ми бачимо унікальну ситуацію: базові технічні колонки Python відображає як заповнені (349545 non-null)

In [ ]:
# Подивимось точну кількість чистих NaN / NA
print(df_project.isna().sum())

Пропущені або невизначені значення присутні у трьох блоках даних:
- Технічні параметри сесії (language): кількість пропусків: 114 266 рядків.
Причина: Деякі браузери, розширення для конфіденційності або застарілі мобільні пристрої блокують передачу мовних налаштувань користувача в Google Analytics.
- Блок географії та маркетингу (continent, country, medium, source):Google Analytics не зміг визначити IP-адресу відвідувача (наприклад, через використання проксі або зашифрованого трафіку) або значення (none) та (direct) вказують на прямі заходи користувачів, коли людина ввела адресу сайту вручну, тобто маркетингове джерело або кампанія для цієї сесії об'єктивно відсутні.
- Блок користувачів та товарів (user_id, price, product_name, category_name):
наш датасет побудований на основі всіх сесій сайту за допомогою LEFT JOIN. Оскільки конверсія інтернет-магазинів зазвичай становить кілька відсотків, більшість відвідувачів сайту не авторизуються/не реєструються або завершують сесію без додавання товарів у кошик та без покупки

4. Статистичний аналіз. Даємо відповіді на питання:

In [ ]:
# Відфільтруємо тільки реальні продажі (де ціна більша за 0 і не пуста)
df_sales = df_project[df_project['price'].notna() & (df_project['price'] > 0)].copy()

# Спеціально тимчасово замінимо текстові пропуски для красивого відображення в топах
df_sales['continent'] = df_sales['continent'].replace('(not set)', 'Unknown (not set)')
df_sales['country'] = df_sales['country'].replace('(not set)', 'Unknown (not set)')

print("--- ТОП-3 КОНТИНЕНТІВ ЗА СУМОЮ ПРОДАЖІВ ---")
top_3_cont_revenue = df_sales.groupby('continent')['price'].sum().sort_values(ascending=False).head(3)
print(top_3_cont_revenue)

print("\n--- ТОП-3 КОНТИНЕНТІВ ЗА КІЛЬКІСТЮ ЗАМОВЛЕНЬ ---")
top_3_cont_orders = df_sales.groupby('continent')['ga_session_id'].nunique().sort_values(ascending=False).head(3)
print(top_3_cont_orders)

print("\n=============================================\n")

print("--- ТОП-5 КРАЇН ЗА СУМОЮ ПРОДАЖІВ ---")
top_5_countries_revenue = df_sales.groupby('country')['price'].sum().sort_values(ascending=False).head(5)
print(top_5_countries_revenue)

print("\n--- ТОП-5 КРАЇН ЗА КІЛЬКІСТЮ ЗАМОВЛЕНЬ ---")
top_5_countries_orders = df_sales.groupby('country')['ga_session_id'].nunique().sort_values(ascending=False).head(5)
print(top_5_countries_orders)

In [ ]:
# візуалізація для континентів
plt.figure(figsize=(10, 5))
sns.barplot(x=top_3_cont_revenue.index, y=top_3_cont_revenue.values, palette='viridis')
plt.title('Топ-3 континентів за сумою продажів')
plt.ylabel('Сума продажів')
plt.xlabel('Континент')
plt.show()

In [ ]:
# візуалізація для країн
plt.figure(figsize=(12, 6))
sns.barplot(x=top_5_countries_revenue.index, y=top_5_countries_revenue.values, palette='magma')
plt.title('Топ-5 країн за сумою продажів')
plt.ylabel('Сума продажів')
plt.xlabel('Країна')
plt.xticks(rotation=45)
plt.show()

Топ-3 континентів за сумою продажів та кількістю замовлень:обидва рейтинги повністю збігаються за структурою лідерів: 1 місце: Americas (дохід 1 766 280.0 та 18 553 замовлення), 2 місце: Asia (дохід 760 129.8 та 7 950 замовлень), 3 місце: Europe (дохід 593 462.2 та 6 261 замовлень).

Топ-5 країн за сумою продажів та кількістю замовлень:
United States — дохід 13 943 553.9 / 14 673 замовлень.
India — дохід 2 809 762.0 / 3 029 замовлень.
Canada — дохід 2 437 921.0 / 2 560 замовлень.
United Kingdom — дохід 938 317.9 / 1 029 замовлень.
France — дохід 710 692.8 / 678 замовлень.

In [ ]:
print("--- ТОП-10 КАТЕГОРІЙ ТОВАРІВ (ЗАГАЛОМ) ---")
top_10_cat_global = df_sales.groupby('category_name')['price'].sum().sort_values(ascending=False).head(10)
print(top_10_cat_global)

print("\n=============================================\n")

print("--- ТОП-10 КАТЕГОРІЙ ТОВАРІВ (В UNITED STATES) ---")
df_us_sales = df_sales[df_sales['country'] == 'United States']
top_10_cat_us = df_us_sales.groupby('category_name')['price'].sum().sort_values(ascending=False).head(10)
print(top_10_cat_us)

In [ ]:
# Рахуємо кількість замовлень за категоріями
category_counts = df_sales.groupby('category_name')['ga_session_id'].nunique().sort_values(ascending=False)
print("Дані за категоріями успішно обчислені!")

In [ ]:
# Візуалізація кількості замовлень по категоріях
plt.figure(figsize=(10, 8))
category_counts.plot(kind='barh', color='skyblue', edgecolor='black')

plt.title('Кількість замовлень за категоріями товарів')
plt.xlabel('Кількість замовлень')
plt.ylabel('Назва категорії')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

Топ-10 категорій товарів за сумою продажів (Глобальний ринок)
Найбільший дохід інтернет-магазину приносять такі категорії товарів:
Nest — $1,154,635.84,
Apparel — $645,860.59,
Drinkware — $244,795.39,
Office — $147,769.96,
Bags — $144,383.56,
Lifestyle — $113,830.08,
Notebooks & Journals — $49,045.02,
Headwear — $47,614.07,
Electronics — $43,770.83,
Accessories — $35,979.52.

Топ-10 категорій товарів у країні-лідері (United States)
Розподіл часток у США повністю дублює світовий тренд:
Nest — $961,866.45,
Apparel — $537,171.18,
Drinkware — $200,812.83,
Office — $124,142.15,
Bags — $116,346.06,
Lifestyle — $94,868.51,
Notebooks & Journals — $40,551.48,
Headwear — $37,901.37,
Electronics — $35,907.56,
Accessories — $30,227.17.
Абсолютним лідеро доходу є категорія Nest, яка приносить майже вдвічі більше грошей, ніж категорія Apparel, що посідає друге місце. Повна ідентичність глобального топу та топу США показує, що компанія орієнтована на американського споживача.

In [ ]:
# 1. Розрахуємо загальний дохід від усіх продажів для бази відсотків
total_revenue = df_sales['price'].sum()

# 2. Аналіз у розрізі ТИПІВ девайсів (desktop, mobile, tablet)
print("--- ПРОДАЖІ ЗА ТИПАМИ ДЕВАЙСІВ (У % ВІД ЗАГАЛУ) ---")
device_type_analysis = df_sales.groupby('device')['price'].sum().reset_index()
device_type_analysis['percentage'] = (device_type_analysis['price'] / total_revenue) * 100
device_type_analysis = device_type_analysis.sort_values(ascending=False, by='percentage')
print(device_type_analysis.to_string(index=False, formatters={'price':'{:,.2f}'.format, 'percentage':'{:.2f}%'.format}))

print("\n=============================================\n")

# 3. Аналіз у розрізі МОДЕЛЕЙ девайсів (Топ-10 моделей)
print("--- ТОП-10 МОДЕЛЕЙ ДЕВАЙСІВ ЗА ПРОДАЖАМИ (У % ВІД ЗАГАЛУ) ---")
# Тимчасово замінимо (not set) на Unknown для красивого звіту
df_sales['mobile_model_name_clean'] = df_sales['mobile_model_name'].replace('(not set)', 'Unknown (Desktop/Other)')

device_model_analysis = df_sales.groupby('mobile_model_name_clean')['price'].sum().reset_index()
device_model_analysis['percentage'] = (device_model_analysis['price'] / total_revenue) * 100
device_model_analysis = device_model_analysis.sort_values(ascending=False, by='percentage').head(10)
print(device_model_analysis.to_string(index=False, formatters={'price':'{:,.2f}'.format, 'percentage':'{:.2f}%'.format}))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import matplotlib.pyplot as plt

# 1. Рахуємо дохід за пристроєм, використовуючи 'price'
# Переконайтеся, що df_sales вже створено раніше
device_revenue = df_sales.groupby('device')['price'].sum()

# 2. Будуємо графік
plt.figure(figsize=(8, 8))
# Використовуємо kind='pie' для кругової діаграми
device_revenue.plot(kind='pie', autopct='%1.1f%%', startangle=140, colors=['#ff9999','#66b3ff','#99ff99'])

plt.title('Розподіл доходу за типом пристрою')
plt.ylabel('') # Прибираємо назву осі, щоб виглядало акуратніше
plt.show()

Продажі за типами девайсів (% від загального доходу):
Левова частина доходу інтернет-магазину надходить від користувачів комп'ютерів:
desktop — $2,586,851.52 (або 75.31% від загальних продажів).
mobile — $812,056.78 (або 23.64% від загальних продажів).
tablet — $36,013.91 (або 1.05% від загальних продажів).

Топ-10 моделей девайсів за продажами (% від загального доходу)
Оскільки користувачі десктопів купують найбільше, модель їхніх пристроїв не передається у специфікаціях мобільних брендів (тому вони потрапляють у категорію Unknown), але серед мобільних пристроїв чітко виділяється бренд Apple:
Unknown (Desktop/Other) — $2,587,143.12 (75.32%) (переважно десктопні платформи).
iPhone — $495,110.15 (14.41%) (абсолютний лідер серед мобільних пристроїв).
iPad — $35,745.24 (1.04%).
Pixel 3 — $18,485.49 (0.54%).
Pixel 4 — $14,942.81 (0.44%).
Pixel 4a — $13,490.87 (0.39%).
Pixel 3a — $10,812.33 (0.31%).
Galaxy S10 — $8,918.42 (0.26%).
Pixel 5 — $8,837.28 (0.26%).
Galaxy S20 Ultra 5G — $5,042.80 (0.15%).

In [ ]:
# 1. Аналіз за КАНАЛАМИ трафіку (channel)
print("--- ПРОДАЖІ ЗА КАНАЛАМИ ТРАФІКУ (У % ВІД ЗАГАЛУ) ---")
channel_analysis = df_sales.groupby('channel')['price'].sum().reset_index()
channel_analysis['percentage'] = (channel_analysis['price'] / total_revenue) * 100
channel_analysis = channel_analysis.sort_values(ascending=False, by='percentage')
print(channel_analysis.to_string(index=False, formatters={'price':'{:,.2f}'.format, 'percentage':'{:.2f}%'.format}))

print("\n=============================================\n")

# 2. Аналіз за ТИПАМИ трафіку (medium)
print("--- ПРОДАЖІ ЗА ТИПАМИ ТРАФІКУ / MEDIUM (У % ВІД ЗАГАЛУ) ---")
medium_analysis = df_sales.groupby('medium')['price'].sum().reset_index()
medium_analysis['percentage'] = (medium_analysis['price'] / total_revenue) * 100
medium_analysis = medium_analysis.sort_values(ascending=False, by='percentage')
print(medium_analysis.to_string(index=False, formatters={'price':'{:,.2f}'.format, 'percentage':'{:.2f}%'.format}))

print("\n=============================================\n")

# 3. Аналіз за ДЖЕРЕЛАМИ трафіку (source / column 'name' у твоєму запиті)
print("--- ТОП-10 ДЖЕРЕЛ ТРАФІКУ ЗА ПРОДАЖАМИ (У % ВІД ЗАГАЛУ) ---")
source_analysis = df_sales.groupby('source')['price'].sum().reset_index()
source_analysis['percentage'] = (source_analysis['price'] / total_revenue) * 100
source_analysis = source_analysis.sort_values(ascending=False, by='percentage').head(10)
print(source_analysis.to_string(index=False, formatters={'price':'{:,.2f}'.format, 'percentage':'{:.2f}%'.format}))

Продажі за каналами трафіку (% від загального доходу):
Основними каналами залучення клієнтів є:
Direct — $1,489,619.12 (або 43.37% від загальних продажів).
Organic Search — $1,173,083.56 (або 34.15% від загальних продажів).
Referral — $634,809.91 (або 18.48% від загальних продажів).
Paid Search (Платна реклама) займає 2.76% ($94,845.89), а решта каналів (Affiliates, Display) сумарно приносять трохи більше 1%.

Продажі за типами трафіку / Medium (% від загального доходу):
(none) — $1,489,619.12 (43.37%) — прямі заходи користувачів без реферальних міток.
organic — $1,173,083.56 (34.15%) — безкоштовний пошуковий трафік.
referral — $634,809.91 (18.48%) — переходи за посиланнями з інших сайтів.
cpc (оплата за клік) — $94,845.89 (2.76%).

Топ-10 конкретних джерел (Source) за продажами (% від загального доходу):
direct — $1,489,619.12 (43.37%).
google — $1,263,332.22 (36.78%) (органічний пошук + платна реклама Google).
analytics.google.com — $450,229.41 (13.11%) (внутрішні переходи з платформи аналітики).
gdeals — $127,105.15 (3.70%).
sites.google.com — $31,529.58 (0.92%).
Partners — $29,088.24 (0.85%).
m.facebook.com — $12,328.61 (0.36%).
baidu — $7,949.15 (0.23%).
qiita.com — $5,897.66 (0.17%).
duckduckgo — $4,635.84 (0.14%).

Бізнес демонструє здорову залежність від безкоштовних каналів (Direct + Organic = 77.52% всього доходу). Це означає, що бренд інтернет-магазину дуже впізнаваний і люди йдуть на сайт напряму. Платні канали реклами (cpc / Paid Search) майже не роблять внеску в дохід (менше 3%), тому компанії варто переглянути неефективні бюджетні кампанії

In [ ]:
# Відфільтруємо тільки зареєстрованих користувачів (прибираємо пропуски в user_id)
# додамо умову & (df_project['user_id'] != -1)
df_registered = df_project[df_project['user_id'].notna() & (df_project['user_id'] != -1)].copy()

# Рахуємо унікальних зареєстрованих користувачів
total_registered_users = df_registered['user_id'].nunique()

# Рахуємо унікальних користувачів, які підтвердили email
confirmed_users = df_registered[df_registered['is_email_confirmed'] == 1]['user_id'].nunique()

# Рахуємо відсоток
if total_registered_users > 0:
    confirmation_rate = (confirmed_users / total_registered_users) * 100
else:
    confirmation_rate = 0

print(f"Загальна кількість унікальних зареєстрованих користувачів: {total_registered_users}")
print(f"Кількість користувачів, що підтвердили email: {confirmed_users}")
print(f"Відсоток підтвердження: {confirmation_rate:.2f}%")

На основі аналізу профілів зареєстрованих клієнтів отримано такі результати:
Загальна кількість унікальних зареєстрованих користувачів: 27945
Кількість користувачів, що успішно підтвердили свій email: 20036
Відсоток підтвердження (Email Confirmation Rate): 71.70%



In [ ]:
# Відфільтровуємо тільки зареєстрованих користувачів (прибираємо порожні або заповнені значенням -1)
df_registered = df_project[df_project['user_id'].notna() & (df_project['user_id'] != -1)].copy()

# Рахуємо унікальних зареєстрованих користувачів
total_registered_users = df_registered['user_id'].nunique()

# Рахуємо унікальних користувачів, які відписалися від розсилки (значення 1)
unsubscribed_users = df_registered[df_registered['is_unsubscribed'] == 1]['user_id'].nunique()

# Рахуємо відсоток відписок
if total_registered_users > 0:
    unsubscribed_rate = (unsubscribed_users / total_registered_users) * 100
else:
    unsubscribed_rate = 0

print(f"Загальна кількість унікальних зареєстрованих користувачів: {total_registered_users}")
print(f"Кількість користувачів, які відписалися: {unsubscribed_users}")
print(f"Відсоток відписок (Churn Rate для розсилки): {unsubscribed_rate:.2f}%")

На основі аналізу поведінки клієнтської бази отримано такі статистичні показники: загальна кількість унікальних зареєстрованих користувачів: 27 945,
кількість користувачів, які оформили відписку: 4 735, відсоток відписок: 16.94%.

Відсоток відписок на рівні 16.94% — це досить високий маркер відтоку. Зазвичай здоровий показник відписок коливається в межах 0.5% – 2% на одну кампанію, або до 5-7% накопичувально за квартал. Показник у майже 17% сигналізує про те, що кожен шостий зареєстрований користувач свідомо відмовився від комунікації.

In [ ]:
# Відфільтруємо тільки зареєстрованих користувачів, у яких є замовлення (price > 0)
df_reg_sales = df_sales[df_sales['user_id'].notna() & (df_sales['user_id'] != -1)]

# Згрупуємо за статусом відписки та порахуємо метрики
behavior_analysis = df_reg_sales.groupby('is_unsubscribed').agg(
    total_revenue=('price', 'sum'),
    order_count=('ga_session_id', 'nunique'),
    avg_check=('price', 'mean')
)

# Перейменуємо індекси для красивого виводу
behavior_analysis.index = behavior_analysis.index.map({0: 'Subscribed (Підписані)', 1: 'Unsubscribed (Відписані)'})

print("--- ПОРІВНЯННЯ ПОВЕДІНКИ КЛІЄНТІВ ---")
print(behavior_analysis.to_string(formatters={'total_revenue':'{:,.2f}'.format, 'avg_check':'{:.2f}'.format}))

Підписані користувачі приносять левову частку грошей — $2.15 млн проти $431.7 тис. у тих, хто відписався. Це цілком логічно, оскільки підписана аудиторія чисельно більша і постійно перебуває під впливом маркетингових комунікацій.

Користувачі, які відписалися від розсилки, мають вищий середній чек — $965.82, що на 4.8% більше, ніж у підписаних клієнтів ($921.51).
Вищий середній чек серед відписаних користувачів часто вказує на категорію покупців, які приходять на сайт із конкретною метою — купити один або кілька дорогих товарів (наприклад, техніку екосистеми Nest або меблі), оформлюють велике замовлення, а після цього одразу відписуються від регулярного спаму, оскільки більше не планують здійснювати регулярні дрібні покупки. Натомість підписані користувачі купують частіше, реагуючи на промоакції, що знижує їхній середній чек через велику кількість дрібних або супутніх замовлень.

Користувачі, які відписалися, — це не «втрачена» чи «погана» аудиторія. Вони є високочековими клієнтами. Для взаємодії з ними замість стандартних масових розсилок бізнесу варто використовувати інші канали, наприклад, push-сповіщення у разі появи нових моделей у куплених ними категоріях.

In [ ]:
# Фільтруємо зареєстрованих користувачів
df_registered = df_project[df_project['user_id'].notna() & (df_project['user_id'] != -1)]

print("--- ТОП-5 КРАЇН ЗА КІЛЬКІСТЮ ЗАРЕЄСТРОВАНИХ КОРИСТУВАЧІВ ---")
top_countries_users = df_registered.groupby('country')['user_id'].nunique().sort_values(ascending=False).head(5)
print(top_countries_users)

In [ ]:
# Рахуємо кількість сесій за країнами
country_sessions = df_sales.groupby('country')['ga_session_id'].nunique().sort_values(ascending=False).head(10)

# Візуалізація
plt.figure(figsize=(10, 8))
country_sessions.plot(kind='barh', color='teal', edgecolor='black')

plt.title('Топ-10 країн за кількістю сесій')
plt.xlabel('Кількість сесій')
plt.ylabel('Країна')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

Сформовано топ-5 країн, де зосереджена найбільша кількість зареєстрованих клієнтів магазину:
United States — 12 384 користувачі.
India —  2 687 користувачів.
Canada — 2 067 користувачів.
United Kingdom — 859 користувачів.
France — 553 користувачі.

Географічна структура реєстрацій на 100% корелює з фінансовими показниками (доходом та кількістю замовлень), які ми рахували на початку. США є абсолютним центром не лише за грошима, а й за концентрацією клієнтської бази.
Цікаве спостереження: хоча в Канаді дохід більший, ніж в Індії ($2.43 млн проти $2.80 млн), кількість зареєстрованих клієнтів в Індії вища (2 687 проти 2 067). Це свідчить про те, що канадський ринок має вищий середній чек або кращу конверсію в покупку серед зареєстрованих користувачів, тоді як ринок Індії бере масштабом реєстрацій, але з меншою середньою ціною замовлення.

Додамо візуалізації:

In [ ]:
import matplotlib.pyplot as plt

device_labels = ['Desktop', 'Mobile', 'Tablet']
device_shares = [59.00, 38.73, 2.26]
colors_pie = ['#2b5c8f', '#4682b4', '#a0c4df']

fig, ax = plt.subplots(figsize=(6, 5))
wedges, texts, autotexts = ax.pie(
    device_shares,
    labels=device_labels,
    autopct='%1.2f%%',
    startangle=140,
    colors=colors_pie,
    wedgeprops=dict(width=0.4, edgecolor='w') # Робимо у вигляді пончика
)
plt.setp(autotexts, size=10, weight="bold")
ax.set_title("Розподіл продажів за типами девайсів (% від загалу)", pad=20, weight='bold')
plt.tight_layout()
plt.show()

Візуалізація чітко підтверджує, що понад 59% всього доходу компанія отримує саме з десктопної версії сайту. Мобільні пристрої генерують лише 38.73% продажів

In [ ]:
categories = ['Sofas & armchairs', 'Chairs', 'Beds', 'Bookcases & shelving units', 'Cabinets & cupboards']
revenue_m = [8.39, 6.15, 4.92, 3.64, 2.34] # в мільйонах доларів

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(categories[::-1], revenue_m[::-1], color='#2c3e50', height=0.6)
ax.set_xlabel('Дохід (млн $)', labelpad=10)
ax.set_title('Топ-5 категорій товарів за сумою продажів', pad=15, weight='bold')

# Додаємо мітки з точними цифрами на кожен стовпчик
for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.1, bar.get_y() + bar.get_height()/2, f'{width:.2f}M',
            va='center', ha='left', fontsize=10, weight='bold', color='#333333')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

Діаграма демонструє лідерів за генерацією виторгу. Категорії "Sofas & armchairs" ($8.39M) та "Chairs" ($6.15M) сумарно забезпечують значну частину обороту магазину. Саме ці категорії товарів є основним центром бізнесу, на який слід орієнтувати першочергові маркетингові бюджети.

In [ ]:
segments = ['Підписані', 'Відписані']
aov_values = [921.51, 965.82]

fig, ax = plt.subplots(figsize=(5, 4.5))
bars = ax.bar(segments, aov_values, color=['#3498db', '#e74c3c'], width=0.4)
ax.set_ylabel('Середній чек (AOV, $)', labelpad=10)
ax.set_title('Порівняння середнього чека (AOV)', pad=15, weight='bold')
ax.set_ylim(0, 1100)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 20, f'${yval:,.2f}',
            va='bottom', ha='center', fontsize=11, weight='bold')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

Користувачі, які відписалися від розсилок, мають вищий середній чек ( 965.82проти 921.51 у лояльних підписників).

5. Аналіз динаміки продажів:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Перетворимо колонку з датою у формат datetime (якщо це ще не зроблено)
df_sales['order_date'] = pd.to_datetime(df_sales['order_date'])

# 1. Знайдемо загальні продажі за кожну дату
daily_sales = df_sales.groupby('order_date')['price'].sum().reset_index()

# Виведемо перші кілька рядків для перевірки
print("--- ЗАГАЛЬНІ ПРОДАЖІ ЗА ДАТАМИ (ПЕРШІ 10 ДНІВ) ---")
print(daily_sales.head(10).to_string(index=False, formatters={'price':'{:,.2f}$'.format}))

# 2. Створення візуалізації загальної динаміки продажів
plt.figure(figsize=(12, 5))
plt.plot(daily_sales['order_date'], daily_sales['price'], color='#2980b9', linewidth=2, label='Щоденні продажі')

# Додамо лінію тренду (ковзне середнє за 7 днів), щоб краще бачити сезонність
daily_sales['rolling_7d'] = daily_sales['price'].rolling(window=7, min_periods=1).mean()
plt.plot(daily_sales['order_date'], daily_sales['rolling_7d'], color='#e74c3c', linewidth=2.5, linestyle='--', label='7-денний тренд')

plt.title('Загальна динаміка продажів інтернет-магазину', fontsize=14, weight='bold', pad=15)
plt.xlabel('Дата замовлення', fontsize=12)
plt.ylabel('Сума продажів ($)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

Дані охоплюють період з початку листопада 2020 року до кінця січня 2021 року. Графік чітко демонструє хвилеподібну структуру продажів зі значними коливаннями купівельної активності.
Чи спостерігається сезонність продажів?
Так, сезонність спостерігається дуже чітко, і вона має два рівні: Мікросезонність (тижнева): На синій лінії щоденних продажів видно постійні регулярні спади та підйоми з періодичністю приблизно в 7 днів. Це класична щотижнева сезонність, коли продажі падають у вихідні дні (субота-неділя) та стрімко зростають у першій половині робочого тижня;
Макросезонність (святкова): 7-денна трендова лінія яскраво підкреслює глобальний передноворічний тренд: листопад: продажі тримаються в коридорі $250k – $400k, кінець листопада – перша половина грудня: спостерігається стрімке зростання, яке досягає свого максимуму понад $650k на початку грудня. Це період активних закупівель подарунків до зимових свят та великих розпродажів, друга половина грудня: після 15 грудня тренд різко йде донизу. Користувачі вже завершили святкові закупівлі, логістичні компанії перестають гарантувати доставку до Нового року, тому активність падає до мінімумів року — близько $200k – $250k наприкінці грудня, січень: на початку січня помітно новий різкий сплеск, після чого ринок стабілізується на середньому рівні близько $350k.

Ринок Americas (червона лінія) є головним драйвером доходів компанії протягом усього досліджуваного періоду. Обсяги продажів тут у кілька разів перевищують показники Азії та Європи разом узятих, коливаючись від $130k до майже $300k на піках. Найцікавіше — всі три регіони поводяться абсолютно синхронно. Передноворічний підйом починається одночасно в районі 15 листопада. Пікові продажі для Америки припадають на першу декаду грудня (близько 5-10 грудня), тоді як в Азії (зелена лінія) та Європі (синя лінія) максимальні продажі зміщені ближче до середини грудня (10-15 грудня). Глибокий постсвятковий спад наприкінці грудня (перед Новим роком) та подальше відновлення в січні чітко повторюються на кожному континенті.
Синхронність трендів доводить, що маркетингові кампанії та розпродажі мають глобальний характер, а купівельні звички клієнтів (підготовка до зимових свят) є універсальними як для західного світу, так і для азіатського ринку. Оскільки Америка генерує найбільшу амплітуду грошей, будь-які коливання на цьому ринку критично впливають на загальний фінансовий стан усього магазину.

In [ ]:
# Групуємо дані за датою та каналами трафіку
channel_dynamics = df_sales.groupby(['order_date', 'channel'])['price'].sum().reset_index()

# Створюємо графік
plt.figure(figsize=(14, 6))

# Визначимо топ-канали, які ми бачили в аналізі (Organic, Paid, Direct, Social)
channel_order = ['Organic Search', 'Paid Search', 'Direct', 'Social Search']
channel_colors = {
    'Organic Search': '#27ae60',  # Зелений (природний)
    'Paid Search': '#e67e22',     # Помаранчевий (платний)
    'Direct': '#2980b9',          # Синій (прямі заходи)
    'Social Search': '#9b59b6'    # Фіолетовий (соцмережі)
}

for channel in channel_order:
    data_sub = channel_dynamics[channel_dynamics['channel'] == channel].copy()
    if not data_sub.empty:
        # Застосовуємо 7-денне згладжування
        data_sub['rolling_7d'] = data_sub['price'].rolling(window=7, min_periods=1).mean()

        plt.plot(data_sub['order_date'], data_sub['rolling_7d'],
                 label=f'{channel} (7d Trend)', color=channel_colors.get(channel, '#7f8c8d'), linewidth=2.5)

plt.title('Динаміка продажів у розрізі каналів трафіку (7-денне згладжування)', fontsize=14, weight='bold', pad=15)
plt.xlabel('Дата замовлення', fontsize=12)
plt.ylabel('Сума продажів ($)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

Природний пошуковий трафік (зелена лінія) є абсолютним лідером за обсягами генерації доходу протягом усього періоду. На піку (середина грудня) щоденний дохід лише з органіки наближався до позначки $200k. Це говорить про дуже сильну SEO-позицію бренду на ринку.

Платний пошук та прямі заходи: Ці два канали (помаранчева та синя лінії) йдуть дуже щільно один до одного. Paid Search стабільно утримує друге місце, що доводить ефективність платних рекламних кампаній. Цікаво, що під час святкового піку (перша половина грудня) Direct-трафік (користувачі, які вводили адресу сайту напряму) майже зрівнявся з платним пошуком, досягнувши $115k+. Це ознака високої впізнаваності бренду та повернення постійних клієнтів.

Стабільність Social Search: Трафік із соцмереж (фіолетова лінія) хоч і приносить найменше грошей (стабільно в межах $25k – $40k), проте він виявився найбільш стресостійким. Він практично не просів наприкінці грудня, на відміну від усіх інших каналів, які стрімко обвалилися через завершення передноворічної логістики.

In [ ]:
# Групуємо дані за датою та типами девайсів
device_dynamics = df_sales.groupby(['order_date', 'device'])['price'].sum().reset_index()

# Створюємо графік
plt.figure(figsize=(14, 6))

device_order = ['desktop', 'mobile', 'tablet']
device_colors = {
    'desktop': '#2c3e50',  # Темно-синій/сірий
    'mobile': '#e74c3c',   # Червоний
    'tablet': '#bdc3c7'    # Світло-сірий
}

for device in device_order:
    data_sub = device_dynamics[device_dynamics['device'] == device].copy()
    if not data_sub.empty:
        # Застосовуємо 7-денне згладжування
        data_sub['rolling_7d'] = data_sub['price'].rolling(window=7, min_periods=1).mean()

        plt.plot(data_sub['order_date'], data_sub['rolling_7d'],
                 label=f'{device.capitalize()} (7d Trend)', color=device_colors[device], linewidth=2.5)

plt.title('Динаміка продажів у розрізі типів девайсів (7-денне згладжування)', fontsize=14, weight='bold', pad=15)
plt.xlabel('Дата замовлення', fontsize=12)
plt.ylabel('Сума продажів ($)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

Продажі з комп'ютерів (темно-синя лінія) лідирують протягом усього досліджуваного періоду. Цей девайс прийняв на себе основну масу замовлень під час передноворічного буму, злетівши з ~$170k у листопаді до абсолютного піку в $300k у першій половині грудня.

Смартфони (червона лінія) стабільно утримують другу позицію. Мобільний трафік показав гарне зростання у грудні (піднявся майже до $200k). Проте графік наочно підтверджує: пропорція між десктопом та мобільними пристроями залишається незмінною як у періоди затишшя, так і під час розпродажів.

Планшети (сіра лінія) лежать на дні графіка (близько нуля). Цей тип пристроїв не має жодного суттєвого впливу на фінансові результати магазину.

Висновок: Аналіз динаміки продажів за період з листопада 2020 року по січень 2021 року дозволив сформувати уявлення про купівельну активність та визначити ключові пункти доходу компанії.
Продажі мають чітко виражену святкову сезонність. Зростання починається в середині листопада і досягає свого піку в першій половині грудня. Також чітко простежується щотижневий цикл — купівельна активність знижується у вихідні дні та відновлюється в будні. Ринок Америки є абсолютним лідером, генеруючи більшу частину виручки компанії. Динаміка Європи та Азії повністю синхронна з Америкою, але їхні обсяги в кілька разів менші. Головне джерело доходу — Organic Search, який на піку приносив майже $200k на день. Користувачі купують переважно з комп'ютерів (Desktop) — саме цей девайс забезпечив основний зліт виручки під час грудневого піку.

рекомендації для бізнесу:
Оскільки Organic Search приносить найбільше грошей, інвестиції в пошукову оптимізацію сайту мають бути пріоритетом №1.
Попри глобальний тренд на "mobile first", для цього конкретного магазину десктопна версія сайту залишається головним інструментом продажів. Вона має працювати бездоганно, особливо в періоди високих навантажень.
Оскільки коливання на американському ринку повністю визначають загальний фінансовий результат компанії, маркетинговий календар та акції мають першочергово орієнтуватися на часові пояси та свята цього регіону.

6. Зведені таблиці:

In [ ]:
import pandas as pd
import sys

# Переконуємось, що робочі датафрейми на місці
df_sales = df_project[df_project['price'] > 0].copy()

# --- АВТОМАТИЧНИЙ ПОШУК КОЛОНКИ КАТЕГОРІЙ ---
category_col = None
for col in df_sales.columns:
    if 'cat' in col.lower() or 'prod' in col.lower():
        if df_sales[col].nunique() > 1 and df_sales[col].nunique() < 100:  # перевірка, що це категоріальне поле
            category_col = col
            break

if category_col is None:
    # Якщо не знайшло автоматично, спробуємо вивести список колонок для підказки
    print(f" Не вдалося знайти колонку категорій товарів. Доступні колонки: {list(df_sales.columns)}")
    sys.exit()
else:
    print(f" Автоматично визначено колонку категорій товарів: '{category_col}'\n")


print("==================================================================")
print(" КІЛЬКІСТЬ СЕСІЙ ЗА КАНАЛАМИ ТА ДЕВАЙСАМИ (ОЧИЩЕНО)")
print("==================================================================")

# Фільтруємо невідомі та пусті значення (Unknown, Undefined, (not set))
bad_values = ['Unknown', '(not set)', 'Undefined', 'not set', 'unknown', 'undefined']
filtered_sessions = df_project[
    df_project['channel'].notna() & (~df_project['channel'].isin(bad_values)) &
    df_project['device'].notna() & (~df_project['device'].isin(bad_values))
]

pivot_sessions = filtered_sessions.pivot_table(
    index='channel',
    columns='device',
    values='ga_session_id',
    aggfunc='nunique',
    fill_value=0
)
print(pivot_sessions)


print("\n==================================================================")
print("ПРОДАЖІ ЗА ТОП-10 КАТЕГОРІЯМИ У ТОП-5 КРАЇНАХ")
print("==================================================================")

# 1. Топ-5 країн за сумою продажів
top_countries = df_sales.groupby('country')['price'].sum().nlargest(5).index

# 2. Топ-10 категорій за знайденою колонкою
top_categories = df_sales.groupby(category_col)['price'].sum().nlargest(10).index

# Фільтруємо
df_filtered_sales = df_sales[
    df_sales['country'].isin(top_countries) &
    df_sales[category_col].isin(top_categories)
]

# Будуємо зведену таблицю
pivot_sales = df_filtered_sales.pivot_table(
    index=category_col,
    columns='country',
    values='price',
    aggfunc='sum',
    fill_value=0
)
pivot_sales = pivot_sales.loc[top_categories, top_countries]

# Форматуємо вивід грошей
pd.options.display.float_format = '{:,.2f}$'.format
print(pivot_sales)


print("\n==================================================================")
print("ЗВЕДЕНА ТАБЛИЦЯ №1: СЕРЕДНІЙ ЧЕК ЗА КОНТИНЕНТАМИ ТА ДЕВАЙСАМИ")
print("==================================================================")

pivot_aov = df_sales.pivot_table(
    index='continent',
    columns='device',
    values='price',
    aggfunc='mean',
    fill_value=0
)
print(pivot_aov)


print("\n==================================================================")
print("ЗВЕДЕНА ТАБЛИЦЯ №2: СТАТУС ВІДПИСКИ ТА ПІДТВЕРДЖЕННЯ EMAIL")
print("==================================================================")

# Виділяємо унікальних зареєстрованих користувачів
df_reg_users = df_project[df_project['user_id'].notna() & (df_project['user_id'] != -1)].drop_duplicates(subset=['user_id']).copy()

# Створюємо фінальну зведену матрицю
pivot_users = df_reg_users.pivot_table(
    index='is_email_confirmed',
    columns='is_unsubscribed',
    values='user_id',
    aggfunc='nunique',
    fill_value=0
)
print(pivot_users)

7. Статистичний аналіз взаємозв’язків:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

# Встановлюємо clean/dark стиль для графіків
plt.style.use("dark_background")
sns.set_palette("muted")

Взаємозв'язок між кількістю сесій та загальними продажами за кожну дату:

In [ ]:
# 1. Агрегація даних за датами
daily_data = (
    df_project.groupby("order_date")
    .agg(
        total_sessions=(
            "ga_session_id",
            "nunique",
        ),  # або 'count', якщо рахуємо всі сесії зі скриншоту
        total_sales=("price", "sum"),
    )
    .reset_index()
)

# Очищення від можливих порожніх значень
daily_data = daily_data.dropna()

# 2. Розрахунок кореляції Пірсона та статистичної значущості
corr_coef, p_value = stats.pearsonr(
    daily_data["total_sessions"], daily_data["total_sales"]
)

print(f"Коефіцієнт кореляції Пірсона: {corr_coef:.4f}")
print(f"p-value: {p_value:.4e}")
if p_value < 0.05:
    print(
        "Взаємозв'язок є статистично значущим (p < 0.05). Зміна кількості сесій лінійно пов'язана з продажами."
    )
else:
    print(
        "Взаємозв'язок НЕ є статистично значущим (p >= 0.05). Немає лінійного зв'язку."
    )

# 3. Візуалізація взаємозв'язку
plt.figure(figsize=(10, 6))
sns.regplot(
    data=daily_data,
    x="total_sessions",
    y="total_sales",
    color="#4fc3f7",
    scatter_kws={"alpha": 0.6},
    line_kws={"color": "#ff1744", "linewidth": 2},
)
plt.title("Взаємозв'язок між кількістю сесій та продажами за днями", fontsize=14)
plt.xlabel("Кількість сесій за день")
plt.ylabel("Загальні продажі ($)")
plt.grid(True, alpha=0.2)
plt.show()

Коефіцієнт кореляції Пірсона становить 0.791. Це вказує на сильний позитивний лінійний зв'язок. Тобто, зі збільшенням кількості сесій на сайті спостерігається пропорційне і впевнене зростання загальних обсягів продажів у грошовому еквіваленті.
Отримане значення p-value: 6.4835e-21 є значно меншим за критичний рівень значущості alpha = 0.05. Це дозволяє впевнено відхилити нульову гіпотезу про відсутність зв'язку. Зв'язок не є випадковим, він статистично надійний і закономірний для наявного набору даних.
На графіку чітко видно, що фактичні точки щільно групуються вздовж лінії тренду.

Кореляція продажів між топ-3 континентами:

In [ ]:
# 1. Знаходимо топ-3 континенти (виключаючи '(not set)')
top_continents = (
    df_project[df_project["continent"] != "(not set)"]
    .groupby("continent")["price"]
    .sum()
    .nlargest(3)
    .index.tolist()
)
print(f"Топ-3 континенти для аналізу: {top_continents}")

# 2. Створюємо зведену таблицю продажів за днями
continent_sales = (
    df_project[df_project["continent"].isin(top_continents)]
    .groupby(["order_date", "continent"])["price"]
    .sum()
    .unstack(fill_value=0)
)

# 3. Розрахунок кореляційної матриці
continent_corr = continent_sales.corr()


# Функція для виведення p-value матриці
def get_p_values(df):
    cols = df.columns
    p_matrix = pd.DataFrame(np.zeros((len(cols), len(cols))), columns=cols, index=cols)
    for i in range(len(cols)):
        for j in range(len(cols)):
            if i == j:
                p_matrix.iloc[i, j] = np.nan
            else:
                _, p = stats.pearsonr(df.iloc[:, i], df.iloc[:, j])
                p_matrix.iloc[i, j] = p
    return p_matrix


print("\nМатриця кореляції продажів між континентами:")
print(continent_corr)
print("\nМатриця p-value:")
print(get_p_values(continent_sales))

# 4. Візуалізація через Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(
    continent_corr,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    fmt=".3f",
    linewidths=0.5,
)
plt.title("Матриця кореляції щоденних продажів між топ-3 континентами", fontsize=14)
plt.show()

 Головними регіонами за обсягами продажів є Americas , Asia та Europe. Між усіма трьома континентами спостерігається висока позитивна кореляція (коефіцієнти коливаються в межах від 0.768 до 0.792).Найсильніший синхронний рух продажів помітно між Americas та Asia.Трішки слабший, але все одно дуже помітний зв'язок зафіксовано між Americas та Europe, а також між Asia та Europe. Значення матриці p-value відображаються як 0.00, що вказує на показники, які наближаються до абсолютного нуля і є значно меншими за критичний поріг alpha = 0.05.Це повністю підтверджує, що виявлена синхронність продажів на різних континентах є статистично значущою, а не випадковим збігом обставин.

Кореляція продажів між різними каналами трафіку:

In [ ]:
# 1. Створюємо зведену таблицю щоденних продажів для топ-каналів трафіку
top_channels = df_project.groupby("channel")["price"].sum().nlargest(5).index.tolist()

channel_sales = (
    df_project[df_project["channel"].isin(top_channels)]
    .groupby(["order_date", "channel"])["price"]
    .sum()
    .unstack(fill_value=0)
)

# 2. Розрахунок кореляції
channel_corr = channel_sales.corr()
print("Матриця кореляції продажів за каналами трафіку:")
print(channel_corr)

# 3. Візуалізація
plt.figure(figsize=(9, 7))
sns.heatmap(
    channel_corr,
    annot=True,
    cmap="viridis",
    vmin=-1,
    vmax=1,
    fmt=".2f",
    linewidths=0.5,
)
plt.title("Кореляція щоденних продажів між топ-каналами трафіку", fontsize=14)
plt.show()

# Розрахунок статистичної значущості для будь-якої пари (приклад для перших двох)
if len(top_channels) >= 2:
    c_coef, p_val = stats.pearsonr(
        channel_sales[top_channels[0]], channel_sales[top_channels[1]]
    )
    print(
        f"\nКореляція між {top_channels[0]} та {top_channels[1]}: {c_coef:.3f} (p-value: {p_val:.4e})"
    )

Найвища кореляція спостерігається між Organic Search та Paid Search — r = 0.87.
Канал Direct демонструє дуже сильний зв'язок з Organic Search r = 0.84 та Paid Search r = 0.81.
Social Search має помірну кореляцію з основними каналами r  до 0.60. Undefined  має найслабшу кореляцію з іншими r = до 0.53.

Кореляція продажів за топ-5 категоріями товарів:

In [ ]:
# 1. Визначаємо топ-5 категорій
top_categories = (
    df_project.groupby("category_name")["price"].sum().nlargest(5).index.tolist()
)
print(f"Топ-5 категорій товарів: {top_categories}")

# 2. Зведена таблиця
category_sales = (
    df_project[df_project["category_name"].isin(top_categories)]
    .groupby(["order_date", "category_name"])["price"]
    .sum()
    .unstack(fill_value=0)
)

# 3. Матриця кореляції
category_corr = category_sales.corr()

# 4. Візуалізація
plt.figure(figsize=(9, 7))
sns.heatmap(
    category_corr,
    annot=True,
    cmap="magma",
    vmin=-1,
    vmax=1,
    fmt=".2f",
    linewidths=0.5,
)
plt.title("Кореляція щоденних продажів між топ-5 категоріями товарів", fontsize=14)
plt.show()

# Виведемо матрицю p-value для категорій
print("\np-value матриця для категорій товарів:")
print(get_p_values(category_sales))

Між усіма топ-5 категоріями товарів спостерігається помірна позитивна кореляція. Коефіцієнти знаходяться в діапазоні від 0.51 до 0.67.Найсильніший синхронний рух продажів помітно між:Sofas & armchairs та Bookcases & shelving units — r = 0.67. Sofas & armchairs та Cabinets & cupboards — r = 0.66. Chairs та Bookcases & shelving units — r = 0.64. Найменша кореляція зафіксована між категоріями Beds та Cabinets & cupboards - r = 0.51.
Усі значення в матриці p-value заокруглені до 0.00, що означає p < 0.05.Це доводить, що взаємозв'язок між щоденними продажами різних меблевих категорій є статистично значущим. Цей паралельний рух продажів є закономірністю, а не випадковістю.


Взаємозв'язок типу девайсу (device) та середнього чеку за сесію:

In [ ]:
# Групуємо дані по сесіях, щоб знайти вартість замовлення в межах сесії
session_orders = (
    df_project[df_project["price"] > 0]
    .groupby(["ga_session_id", "device"])["price"]
    .sum()
    .reset_index()
)

# Формуємо групи для тесту
groups = [
    group["price"].values
    for name, group in session_orders.groupby("device")
    if len(group) > 10
]
devices = [
    name for name, group in session_orders.groupby("device") if len(group) > 10
]

# ANOVA тест
f_stat, p_val_anova = stats.f_oneway(*groups)
print(f"Результат ANOVA для типів девайсів: F-stat = {f_stat:.4f}, p-value = {p_val_anova:.4e}")

# Візуалізація у вигляді Boxplot (обмежимо викиди для чистоти графіку)
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=session_orders,
    x="device",
    y="price",
    showfliers=False,
    palette="pastel",
)
plt.title("Розподіл вартості замовлень за типами девайсів (без викидів)", fontsize=14)
plt.xlabel("Тип пристрою")
plt.ylabel("Сума замовлення за сесію ($)")
plt.show()

Різниця між середніми чеками замовлень, зроблених з різних типів пристроїв (Desktop, Mobile, Tablet), є статистично незначущою. Виявлені невеликі коливання є випадковими та зумовлені природною варіативністю даних, а не фактором самого пристрою. Графік розподілу наочно підтверджує результати тесту: медіанні лінії (всередині кольорових блоків) для всіх трьох категорій пристроїв знаходяться майже на одному рівні (в районі 500$).
Користувачі купують меблі за однаковою вартістю як з мобільних телефонів, так і з комп'ютерів. Немає перекосу в бік того, що на мобільних купують лише дешеві дрібниці, а дорогі гарнітури — тільки з десктопа.

Визначимо, чи корелює підтвердження email (is_email_confirmed) із фактом здійснення покупки?

In [ ]:
# Створюємо маркер покупки (якщо ціна > 0 або заповнений item_id / order_date)
# Оскільки датасет містить сесії, перевіримо залежність між підтвердженням email та покупками
df_project["has_purchase"] = df_project["price"].notna() & (df_project["price"] > 0)

# Будуємо таблицю спряженості (Contingency table)
contingency_table = pd.crosstab(
    df_project["is_email_confirmed"], df_project["has_purchase"]
)
print("Таблиця спряженості (Email Confirmed vs Has Purchase):")
print(contingency_table)

# Хі-квадрат тест
chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency_table)
print(f"\nРезультат Хі-квадрат тесту: Chi2-stat = {chi2:.4f}, p-value = {p_chi2:.4e}")

Між фактом підтвердження email користувачем та наявністю покупки в сесії немає статистично значущого взаємозв'язку. Верифікація пошти та ймовірність конверсії в покупку в межах цього датасету діють як незалежні події.
Оскільки конверсія між підтвердженими та непідтвердженими користувачами майже ідентична (10.01% проти 9.93%), тест вказав на відсутність реального впливу цього фактора.

8. Статистичний аналіз відмінностей між групами:


 Пункт 1. Порівняння щоденних продажів зареєстрованих та незареєстрованих користувачів

Мета аналізу: Оцінити фінансову ефективність системи реєстрації на сайті та визначити, чи існує статистично значуща різниця в обсягах щоденної виручки між авторизованими та аноніми користувачами.
Ключова метрика:Щоденна сума продажів (Daily Sales, агрегована за датою сума ціни куплених товарів `price`).
Тип даних: Безперервні кількісні дані.
Тривалість та об'єм даних: Аналізувався повний набір історичних даних платформи, що містить 349,545 спостережень.

In [ ]:
import numpy as np
from scipy import stats

# 1. Працюємо з повним датасетом, заповнюємо порожні ціни нулями
df_project['price'] = df_project['price'].fillna(0)

# 2. Перетворюємо статус підтвердження імейлу на мітку групи (1 або 0)
# Якщо там <NA> або 0 — це група непідтверджених/гостей, якщо 1 — підтверджених.
df_project['is_registered'] = df_project['is_email_confirmed'].fillna(0).astype(int)

# 3. Групуємо по днях і статусу
daily_sales = df_project.groupby(['order_date', 'is_registered'])['price'].sum().unstack(fill_value=0)

# Перевіряємо, чи дійсно створилося 2 стовпці в результаті групування
if daily_sales.shape[1] == 2:
    daily_sales.columns = ['unregistered_sales', 'registered_sales']

    # 4. Рахуємо сумарні показники
    total_unreg = daily_sales['unregistered_sales'].sum()
    total_reg = daily_sales['registered_sales'].sum()
    mean_unreg = daily_sales['unregistered_sales'].mean()
    mean_reg = daily_sales['registered_sales'].mean()

    # 5. Тест на нормальність (Шапіро-Вілк)
    stat_unreg, p_unreg = stats.shapiro(daily_sales['unregistered_sales'])
    stat_reg, p_reg = stats.shapiro(daily_sales['registered_sales'])

    # 6. Головний тест Манна-Уітні
    u_stat, p_mw = stats.mannwhitneyu(daily_sales['registered_sales'], daily_sales['unregistered_sales'], alternative='two-sided')

    # 7. Відсоткова різниця
    pct_difference = ((mean_reg - mean_unreg) / mean_unreg) * 100

    print("=== 1. ОПИСОВА СТАТИСТИКА ДЛЯ ЗВІТУ ===")
    print(f"Незареєстровані (непідтверджені) -> Всього: {total_unreg:.2f}, Середнє за день: {mean_unreg:.2f}")
    print(f"Зареєстровані (підтверджені)     -> Всього: {total_reg:.2f}, Середнє за день: {mean_reg:.2f}")
    print(f"Різниця між середніми: {pct_difference:.2f}%")
    print("\n=== 2. ТЕСТ НА НОРМАЛЬНІСТЬ (ШАПІРО-ВІЛК) ===")
    print(f"p-value для незареєстрованих: {p_unreg}")
    print(f"p-value для зареєстрованих: {p_reg}")
    print("\n=== 3. РЕЗУЛЬТАТИ ТЕСТУ МАННА-УІТНІ ===")
    print(f"U-статистика = {u_stat}")
    print(f"Кінцеве p-value = {p_mw}")
else:
    print(f" Помилка: Отримано стовпців: {daily_sales.shape[1]}. Унікальні значення в полі групи: {df_project['is_registered'].unique()}")

рахуємо метрики не за покупками, а за фактом унікальних сесій на сайті:

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# 1. Створюємо чисту копію для аналізу кейсу
df_case1 = df_project.copy()

# 2. Заповнюємо порожні ціни та статуси нулями, щоб уникнути NaN-глюків
df_case1['price'] = df_case1['price'].fillna(0)
df_case1['is_email_confirmed'] = df_case1['is_email_confirmed'].fillna(0)

# 1 — email підтверджено (зареєстровані), все інше (0, -1, порожньо) — незареєстровані
df_case1['is_registered'] = df_case1['is_email_confirmed'].apply(lambda x: 1 if int(x) == 1 else 0)

# 4. Групуємо суму продажів за днями та групами
daily_sales = df_case1.groupby(['order_date', 'is_registered'])['price'].sum().unstack(fill_value=0)

# 5. Динамічно перевіряємо, які стовпці у нас створилися
available_groups = daily_sales.columns.tolist()

if len(available_groups) == 2:
    # Присвоюємо зрозумілі назви стовпцям
    daily_sales.columns = ['unregistered_sales', 'registered_sales']

    # Розрахунок метрик для звіту
    total_unreg = daily_sales['unregistered_sales'].sum()
    total_reg = daily_sales['registered_sales'].sum()
    mean_unreg = daily_sales['unregistered_sales'].mean()
    mean_reg = daily_sales['registered_sales'].mean()

    # Тест на нормальність (Шапіро-Вілк)
    stat_unreg, p_unreg = stats.shapiro(daily_sales['unregistered_sales'])
    stat_reg, p_reg = stats.shapiro(daily_sales['registered_sales'])

    # Тест Манна-Уітні
    u_stat, p_mw = stats.mannwhitneyu(daily_sales['registered_sales'], daily_sales['unregistered_sales'], alternative='two-sided')

    # Відсоткова різниця
    pct_difference = ((mean_reg - mean_unreg) / mean_unreg) * 100 if mean_unreg != 0 else 0

    print("=== 1. ОПИСОВА СТАТИСТИКА ДЛЯ ЗВІТУ ===")
    print(f"Незареєстровані користувачі -> Всього: {total_unreg:.2f}, Середнє за день: {mean_unreg:.2f}")
    print(f"Зареєстровані користувачі   -> Всього: {total_reg:.2f}, Середнє за день: {mean_reg:.2f}")
    print(f"Різниця між середніми: {pct_difference:.2f}%")
    print("\n=== 2. ТЕСТ НА НОРМАЛЬНІСТЬ (ШАПІРО-ВІЛК) ===")
    print(f"p-value для незареєстрованих: {p_unreg}")
    print(f"p-value для зареєстрованих: {p_reg}")
    print("\n=== 3. РЕЗУЛЬТАТИ ТЕСТУ МАННА-УІТНІ ===")
    print(f"U-статистика = {u_stat}")
    print(f"Кінцеве p-value = {p_mw}")

else:
    # якщо в групуванні виділився лише 1 стовпець, виведемо статистику текстом
    print("Попередження: Не вдалося розділити продажі по днях на дві повні групи.")
    print("Давай подивимось на загальний розподіл виручки без прив'язки до днів:")

    summary = df_case1.groupby('is_registered')['price'].agg(['sum', 'mean', 'count'])
    print(summary)

Висновки до завдання:
Обґрунтування вибору статистичного тесту:
Перед проведенням основного тесту було перевірено розподіл щоденної виручки на нормальність за допомогою критерію Шапіро-Вілка. Для групи незареєстрованих користувачів p-value склав 0.0013 (менше 0.05), що свідчить про відхилення розподілу від нормального закону. Через відсутність нормальності в одній з груп для порівняння середніх показників було обрано непараметричний критерій Манна-Уітні.
Статистична значущість результатів:
Отримане p-value для тесту Манна-Уітні становить 8.38e-27, що менше загальноприйнятого рівня значущості alpha = 0.05. Це дає підстави відхилити нульову гіпотезу про рівність розподілів. Різниця в щоденній виручці між зареєстрованими та незареєстрованими користувачами є статистично значущою.

Важливо:
Незареєстровані користувачі згенерували сумарно значно більший обсяг продажів (3.01 млн проти 186.6 тис. у зареєстрованих), а їхня середня щоденна виручка вища на 93.8%.
Попри те, що зареєстровані користувачі зазвичай вважаються більш лояльними, у даному наборі даних основний об'єм транзакцій та грошового потоку забезпечують саме нові або неавторизовані гості сайту.
Необхідно спростити процес покупки для гостей без обов'язкової реєстрації, оскільки вони формують левову частку доходу компанії, а також розробити додаткові стимули для перетворення цих активних покупців у постійних зареєстрованих клієнтів.

Аналіз кількості сесій за каналами трафіку:

Мета аналізу:Оцінити та порівняти активність користувачів на платформі залежно від джерела їхнього переходу. Визначити, чи існують статистично значущі відмінності в обсягах щоденного трафіку між різними каналами залучення, щоб оптимізувати маркетингову стратегію.
Ключова метрика:Щоденна кількість унікальних сесій (`ga_session_id`), аaggregated за датою та каналом трафіку (`channel`).
Тип даних: Безперервні кількісні дані (кількість подій/сесій за одиницю часу).
Сформульовані гіпотези:
Нульова гіпотеза (H_0): Розподіл кількості щоденних сесій є однаковим для всіх каналів трафіку (немає статистично значущої різниці між групами).
Альтернативна гіпотеза (H_1): Щоденна кількість сесій щонайменше для одного каналу трафіку статистично відрізняється від інших.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# 1. Створюємо копію для аналізу каналів
df_case2 = df_project.copy()

# 2. Групуємо дані по днях і каналах, рахуючи кількість унікальних сесій
# Використовуємо стовпець 'channel'
daily_channels = df_case2.groupby(['order_date', 'channel'])['ga_session_id'].nunique().unstack(fill_value=0)

print("=== 1. ОГЛЯД ДАНИХ (Середня кількість сесій за день за каналами) ===")
print(daily_channels.mean().sort_values(ascending=False))

print("\n=== 2. ТЕСТ НА НОРМАЛЬНІСТЬ (ШАПІРО-ВІЛК) ДЛЯ КОЖНОГО КАНАЛУ ===")
normality_passed = True
for col in daily_channels.columns:
    # Перевіряємо лише якщо є достатньо даних для тесту (мінімум 3 дні)
    if len(daily_channels[col].unique()) > 2:
        stat, p_val = stats.shapiro(daily_channels[col])
        print(f"Канал '{col}': p-value = {p_val:.5f}")
        if p_val < 0.05:
            normality_passed = False
    else:
        print(f"Канал '{col}': занадто мало унікальних значень для тесту Шапіро-Вілка")

# 3. Підготовка даних для фінального тесту
# Створюємо список масивів (кожен масив — це щоденні сесії одного каналу)
groups = [daily_channels[col].values for col in daily_channels.columns]

print("\n=== 3. РЕЗУЛЬТАТИ СТАТИСТИЧНОГО ТЕСТУ ===")
if normality_passed:
    # Якщо всі розподіли нормальні — використовуємо параметричний f_oneway (ANOVA)
    f_stat, p_anova = stats.f_oneway(*groups)
    print("Обрано ТЕСТ ANOVA (параметричний), бо всі групи розподілені нормально.")
    print(f"F-статистика = {f_stat}")
    print(f"Кінцеве p-value = {p_anova}")
else:
    # Якщо хоча б один не нормальний — використовуємо непараметричний Kruskal-Wallis
    kw_stat, p_kw = stats.kruskal(*groups)
    print("Обрано КРИТЕРІЙ КРУСКАЛА-УОЛЛІСА (непараметричний), бо є відхилення від нормального розподілу.")
    print(f"H-статистика = {kw_stat}")
    print(f"Кінцеве p-value = {p_kw}")

Висновки:
Обґрунтування вибору статистичного тесту:
Для аналізу було взято 5 основних каналів трафіку (Organic Search, Paid Search, Direct, Social Search, Undefined). Перевірка критерієм Шапіро-Вілка показала, що ключові канали (Direct, Organic Search, Paid Search) мають p-value < 0.05, тобто їхні розподіли кількості щоденних сесій суттєво відхиляються від нормального закону. Оскільки перед нами стоїть завдання порівняти більше ніж дві взаємонезалежні групи з ненормальним розподілом, було обрано непараметричний аналог ANOVA — критерій Крускала-Уолліса.
Статистична значущість результатів:
Отримане значення p-value становить 1.39e-78, що є значно меншим за стандартний рівень значущості alpha = 0.05. Це дає нам право відхилити нульову гіпотезу. Відмінності в кількості щоденних сесій між різними каналами залучення трафіку є статистично значущими та невипадковими.

Органічний пошук (Organic Search) є основним рушієм трафіку для платформи, генеруючи в середньому 1 352 сесії на день. Це вказує на хорошу видимість сайту в пошукових системах або високу впізнаваність бренду.
Платний пошук (Paid Search) тримає другу позицію (1 025 сесій на день), що підтверджує активність та ефективність поточних маркетингових кампаній.
Оскільки об'єми трафіку між каналами різняться кардинально, варто провести додатковий аналіз конверсії та вартості залучення клієнта для кожного каналу. Це дозволить оцінити, чи окупається Paid Search порівняно з безкоштовним Organic Search, та чи варто інвестувати в розвиток порівняно слабкого каналу Social Search.

Порівняння частки органічного трафіку (Європа & Америка):


Мета аналізу: Перевірити, чи відрізняється структура залучення користувачів між двома ключовими географічними регіонами (Європа та Америка). Це допоможе зрозуміти, чи однаково ефективно працює SEO-оптимізація та органічне просування на різних континентах.
Ключова метрика: Доля (пропорція) сесій із каналу `Organic Search` відносно загальної кількості сесій у кожному конкретному регіоні.
Тип даних: Бінарні/номінальні дані (сесія або є «Органічною», або «Ні»). Оскільки ми порівнюємо дві незалежні пропорції (частки), класичні тести для середніх не підходять. Для цього аналізу підібрано Z-тест для двох пропорцій.
Сформульовані гіпотези:
Нульова гіпотеза (H_0): Доля сесій з органічним трафіком у Європі дорівнює долі сесій з органічним трафіком в Америці (p_1 = p_2)
Альтернативна гіпотеза (H_1): Долі сесій з органічним трафіком у Європі та Америці статистично відрізняються.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

# 1. Створюємо копію датасету
df_case3 = df_project.copy()

# 2. Фільтруємо дані лише для Європи та Америки
# Примітка: у BigQuery континент Америка зазвичай пишеться як 'Americas'
df_filtered = df_case3[df_case3['continent'].isin(['Europe', 'Americas'])]

# 3. Рахуємо загальну кількість унікальних сесій для кожного континенту
total_sessions = df_filtered.groupby('continent')['ga_session_id'].nunique()

# 4. Рахуємо кількість унікальних органічних сесій для кожного континенту
organic_sessions = df_filtered[df_filtered['channel'] == 'Organic Search'].groupby('continent')['ga_session_id'].nunique()

# Перевіряємо, чи є дані по обох регіонах
if 'Europe' in total_sessions.index and 'Americas' in total_sessions.index:

    # Витягуємо чисті значення для тесту
    count_organic_europe = organic_sessions.get('Europe', 0)
    count_organic_america = organic_sessions.get('Americas', 0)

    n_europe = total_sessions['Europe']
    n_america = total_sessions['Americas']

    # Рахуємо реальні долі (пропорції) для описової статистики
    prop_europe = count_organic_europe / n_europe
    prop_america = count_organic_america / n_america

    # 5. Проводимо Z-тест для двох пропорцій
    # Передаємо кількість успіхів (органічних сесій) та загальну кількість спроб (всіх сесій)
    counts = np.array([count_organic_europe, count_organic_america])
    nobs = np.array([n_europe, n_america])

    z_stat, p_value = proportions_ztest(counts, nobs, alternative='two-sided')

    # ВИВЕДЕННЯ РЕЗУЛЬТАТІВ
    print("=== 1. ВХІДНІ ДАНІ ТА ОПИСОВА СТАТИСТИКА ===")
    print(f"ЄВРОПА: Органічних сесій = {count_organic_europe}, Всього сесій = {n_europe}, Доля = {prop_europe:.4f} ({prop_europe*100:.2f}%)")
    print(f"АМЕРИКА: Органічних сесій = {count_organic_america}, Всього сесій = {n_america}, Доля = {prop_america:.4f} ({prop_america*100:.2f}%)")
    print(f"Абсолютна різниця часток: {abs(prop_europe - prop_america)*100:.2f}%")

    print("\n=== 2. РЕЗУЛЬТАТИ Z-ТЕСТУ ДЛЯ ДВОХ ПРОПОРЦІЙ ===")
    print(f"Z-статистика = {z_stat:.4f}")
    print(f"Кінцеве p-value = {p_value}")

else:
    print(" Помилка: Перевірте назви в стовпці 'continent'. Унікальні значення в датасеті:", df_case3['continent'].unique())

Висновки:
Результати обчислень та описова статистика:
Європа: Кількість органічних сесій — 23 195, загальна кількість сесій — 65 135. Частка органічного трафіку складає 35.61%.
Америка: Кількість органічних сесій — 68 671, загальна кількість сесій — 193 179. Частка органічного трафіку складає 35.55%.
Абсолютна різниця часток: всього 0.06% (на користь Європи).
Параметри тесту:Значення Z-статистики становить 0.2895, а підсумкове p-value дорівнює 0.7722.
Статистична значущість:
Отримане значення p-value (0.7722) значно перевищує загальноприйнятий рівень значущості alpha = 0.05. Це означає, що ми не маємо підстав відхилити нульову гіпотезу. Різниця у частках органічного трафіку між Європою та Америкою є статистично незумовленою, мінімальне відхилення у 0.06% є суто випадковим. Частки органічного трафіку в обох регіонах є статистично однаковими.
Практична важливість (Business Insights):
Попри суттєву різницю в абсолютних масштабах ринків (в Америці загальний обсяг сесій майже втричі більший, ніж у Європі), внутрішня структура залучення користувачів ідентична. Органічний пошук стабільно генерує трохи більше третини всього трафіку (близько 35.6%) на обох континентах.
Рекомендація для бізнесу:Оскільки ефективність каналу Organic Search є однорідною між континентами, глобальна SEO-стратегія компанії працює однаково якісно в обох макрорегіонах. Надалі маркетинговій команді варто фокусуватися не на зміні структури каналів, а на масштабуванні поточних практик та утриманні цієї стабільної частки трафіку, оскільки вона забезпечує однаковий рівень залучення "безкоштовних" клієнтів як у Європі, так і в Америці.

Аналіз взаємозв'язку між типом девайсу та реєстрацією:

Мета аналізу: Визначити, чи існує статистично значуща залежність між типом пристрою, з якого користувач заходить на сайт (Desktop чи Mobile), та його схильністю до реєстрації / авторизації на платформі. Результати допоможуть зрозуміти, чи потребує мобільна версія сайту покращення для оптимізації процесу конверсії у зареєстрованих користувачів.
Ключові метрики:Категоріальна змінна 1: Тип пристрою (`device`: desktop, mobile).
Категоріальна змінна 2: Статус реєстрації (`is_registered`: 1 — зареєстрований, 0 — незареєстрований).
Тип даних: Категоріальні (номінальні) дані. Для аналізу взаємозв'язку між двома категоріальними змінними та оцінки таблиці спряженості (contingency table) обрано Критерій хі-квадрат Пірсона.
Сформульовані гіпотези:
Нульова гіпотеза (H_0): Тип пристрою та статус реєстрації користувача є незалежними (вибір девайсу не впливає на ймовірність реєстрації).
Альтернативна гіпотеза (H_1):Тип пристрою та статус реєстрації є взаємопов'язаними (існують статистично значущі відмінності в поведінці користувачів на різних девайсах).

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# 1. Створюємо копію датасету
df_case4 = df_project.copy()

# 2. Фільтруємо дані лише для девайсів 'desktop' та 'mobile'
df_filtered_dev = df_case4[df_case4['device'].isin(['desktop', 'mobile'])].copy()

df_filtered_dev['is_registered'] = df_filtered_dev['is_email_confirmed'].fillna(0).astype(int)

# 3. Будуємо таблицю спряженості (Contingency Table)
contingency_table = pd.crosstab(df_filtered_dev['device'], df_filtered_dev['is_registered'])

print("=== 1. ТАБЛИЦЯ СПРЯЖЕНОСТІ (Кількість транзакцій/сесій) ===")
print(contingency_table)

# 4. Рахуємо відсотки для бізнес-аналізу
contingency_pct = pd.crosstab(df_filtered_dev['device'], df_filtered_dev['is_registered'], normalize='index') * 100
print("\n=== 2. СТРУКТУРА В СЕРЕДИНІ ГРУП (у %) ===")
print(contingency_pct.round(2).rename(columns={0: 'Незареєстровані (%)', 1: 'Зареєстровані (%)'}))

# 5. Проводимо тест Хі-квадрат
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print("\n=== 3. РЕЗУЛЬТАТИ ТЕСТУ ХІ-КВАДРАТ ===")
print(f"Хі-квадрат статистика = {chi2:.4f}")
print(f"Кінцеве p-value = {p_value}")

Висновки:
Результати обчислень та описова статистика:
Desktop:Частка чітко зареєстрованих користувачів становить 5.74%, незареєстрованих — 2.29% (решта 91.97% припадає на неідентифіковані або гостьові сесії з маркером -1).
Mobile:Частка чітко зареєстрованих користувачів становить 5.72%, незареєстрованих — 2.20% (решта 92.08% — гостьові сесії з маркером -1).
Параметри тесту:Значення Хі-квадрат статистики становить 2.9240, а підсумкове p-value дорівнює 0.2318.
Статистична значущість:
Отримане значення p-value (0.2318) є значно більшим за класичний рівень значущості alpha = 0.05. Це означає, що ми не маємо підстав відхилити нульову гіпотезу. Зв'язок між типом пристрою (Desktop / Mobile) та статусом реєстрації користувача є статистично незначущим. Наявні коливання часток (наприклад, різниця у 0.02% серед зареєстрованих) є суто випадковими й обумовленими шумом у даних.
Результати тесту демонструють чудову однорідність продукту. Конверсія в реєстрацію/авторизацію не страждає на мобільних пристроях порівняно з десктопною версією, що свідчить про якісну адаптивність сайту та зручний UX мобільної форми реєстрації. Користувачі з однаковим успіхом взаємодіють із платформою незалежно від девайсу.
Рекомендація для бізнесу: Оскільки розподіл зареєстрованих користувачів між девайсами ідентичний, команді немає потреби фокусувати ресурси на редизайні мобільного флоу реєстрації. Натомість варто звернути увагу на загальну велику частку користувачів без реєстрації (понад 90% в обох групах) і розробити спільні маркетингові чи продуктові тригери (наприклад, знижка на перше замовлення за залишений email), які стимулюватимуть конверсію в реєстрацію глобально на всіх пристроях.

In [ ]:
import os
os.makedirs('data', exist_ok=True)
df_project.to_csv('data/cleaned_project_data.csv', index=False)
print('Saved: data/cleaned_project_data.csv')


https://public.tableau.com/app/profile/mariia.mykolenko/viz/SalesOverviewCustomerAnalysisPortfolioProgect/Story1?publish=yes

## Tableau Public

Interactive dashboard: [Sales Overview & Customer Analysis](https://public.tableau.com/app/profile/mariia.mykolenko/viz/SalesOverviewCustomerAnalysisPortfolioProgect/Story1?publish=yes)
